# Exploration notebook - AI Data Classification System

This notebook walks through the same supervised-learning pipeline as the Streamlit app, one stage at a time,
using the project's own modules (`src/` and `utils/`). Run the cells from top to bottom.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config
from src.data_loader import class_distribution, data_quality_table, load_builtin_iris, summarize_dataset
from src.evaluation import METRIC_EXPLANATIONS
from src.model_training import build_pipeline, train_pipeline
from src.preprocessing import clean_data, select_features_and_target, split_data
from src.workflow import run_experiment
from utils import visualization as viz

## 1. Load and understand the dataset

In [ ]:
dataset = load_builtin_iris()
print(f"Target column: {dataset.target_column}")
print(f"Feature columns: {dataset.feature_columns}")
dataset.frame.head()

In [ ]:
print(summarize_dataset(dataset))
data_quality_table(dataset)

In [ ]:
dataset.frame.describe().T.round(3)

**Observation:** there are no missing values, but the summary reports one exact duplicate row.
Duplicates are removed in preprocessing so the same sample can never appear in both the training and the test set.

## 2. Explore

In [ ]:
viz.plot_class_distribution(class_distribution(dataset))

In [ ]:
viz.plot_feature_distributions(dataset.frame, dataset.feature_columns, dataset.target_column)

In [ ]:
viz.plot_correlation_heatmap(dataset.frame, dataset.feature_columns)

**Observation:** the three classes are perfectly balanced (50 each). Petal length and petal width separate the
classes much more clearly than the sepal measurements, and they are strongly correlated with each other (r ~ 0.96).

## 3. Preprocess, select features and split

In [ ]:
cleaned, report = clean_data(dataset.frame, dataset.feature_columns, dataset.target_column)
print(report)

X, y = select_features_and_target(cleaned, dataset.feature_columns, dataset.target_column)
split = split_data(X, y, test_size=config.DEFAULT_TEST_SIZE, random_state=config.RANDOM_STATE)
print(f"train rows: {len(split.X_train)}, test rows: {len(split.X_test)}, stratified: {split.stratified}")
split.y_test.value_counts()

### Leakage check

Scaling statistics must come from the training set only. The fitted scaler's means should match the **training**
means, not the means of the whole dataset.

In [ ]:
fitted = train_pipeline(build_pipeline("Logistic Regression"), split.X_train, split.y_train)
comparison = pd.DataFrame(
    {
        "scaler (fitted in pipeline)": fitted.named_steps["scaler"].mean_,
        "training set mean": split.X_train.mean().to_numpy(),
        "whole dataset mean": X.mean().to_numpy(),
    },
    index=X.columns,
)
comparison.round(4)

## 4. Train, evaluate and compare

In [ ]:
experiment = run_experiment(dataset, dataset.feature_columns, compare=True, save_path=None)
evaluation = experiment.evaluation
print(f"train accuracy: {evaluation.train_accuracy:.3f}   test accuracy: {evaluation.accuracy:.3f}")
print(f"precision: {evaluation.precision:.3f}   recall: {evaluation.recall:.3f}   F1: {evaluation.f1:.3f}")
evaluation.report_table.round(3)

In [ ]:
viz.plot_confusion_matrix(evaluation.confusion_array, evaluation.class_names)

In [ ]:
for metric, meaning in METRIC_EXPLANATIONS.items():
    print(f"{metric}: {meaning}")

In [ ]:
experiment.comparison.round(3)

### Choosing K for K-Nearest Neighbors

K is the number of neighbours that vote on a class. The sweep below uses cross-validation on the **training** set only.

In [ ]:
from src.model_training import knn_k_sweep, recommended_k

sweep = knn_k_sweep(split.X_train, split.y_train)
best_k = recommended_k(sweep)
print(f"K with the lowest cross-validated error: {best_k}  (the model uses K = {config.KNN_NEIGHBORS})")
viz.plot_k_selection(sweep, config.KNN_NEIGHBORS, best_k)

## 5. How much does the result depend on the split?

The test set has only 30 samples, so a single mistake changes accuracy by 3.3 percentage points.
Repeating the experiment with 50 different random splits shows the realistic spread.

In [ ]:
accuracies = [
    run_experiment(dataset, dataset.feature_columns, random_state=seed, save_path=None).evaluation.accuracy
    for seed in range(50)
]
pd.Series(accuracies, name="test accuracy").describe().round(3)

## 6. Predict a new sample

In [ ]:
from src.prediction import explanation_summary, predict_sample

sample = {"sepal_length": 6.3, "sepal_width": 2.9, "petal_length": 5.6, "petal_width": 1.8}
result = predict_sample(sample, experiment.artifact)
print(explanation_summary(result))
result.explanation.round(3)